In [0]:
from pyspark.sql import functions as F

base_path = "/Volumes/workspace/apex_retail/raw_landing_zone"
catalog = "workspace"
schema = "apex_retail"

# Create the schema if it doesn't exist (should already exist from your Volume setup)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")

DataFrame[]

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {catalog}.{schema}.bronze_customer")
spark.sql(f"DROP TABLE IF EXISTS {catalog}.{schema}.bronze_product")
spark.sql(f"DROP TABLE IF EXISTS {catalog}.{schema}.bronze_sales")
print("Old Bronze tables dropped")

Old Bronze tables dropped


In [0]:
datasets = ["customer", "product", "sales"]
loads = ["historical", "incremental"]

for ds in datasets:
    for load in loads:
        landing_path = f"{base_path}/landing/{ds}/{load}/"
        df = spark.read.parquet(landing_path)

        df_bronze = (df
            .withColumn("ingested_at", F.current_timestamp())
            .withColumn("source_load", F.lit(load)))

        bronze_table = f"{catalog}.{schema}.bronze_{ds}"
        write_mode = "overwrite" if load == "historical" else "append"

        (df_bronze.write
            .format("delta")
            .mode(write_mode)
            .option("mergeSchema", "true")
            .saveAsTable(bronze_table))

        print(f"Bronze write complete: {bronze_table} ({load}) -> {df_bronze.count()} rows")

Bronze write complete: workspace.apex_retail.bronze_customer (historical) -> 1052 rows
Bronze write complete: workspace.apex_retail.bronze_customer (incremental) -> 1053 rows
Bronze write complete: workspace.apex_retail.bronze_product (historical) -> 1043 rows
Bronze write complete: workspace.apex_retail.bronze_product (incremental) -> 1041 rows
Bronze write complete: workspace.apex_retail.bronze_sales (historical) -> 1002 rows
Bronze write complete: workspace.apex_retail.bronze_sales (incremental) -> 1000 rows


In [0]:
for ds in datasets:
    spark.sql(f"SELECT source_load, COUNT(*) AS total_rows FROM {catalog}.{schema}.bronze_{ds} GROUP BY source_load").show()

+-----------+----------+
|source_load|total_rows|
+-----------+----------+
|incremental|      1053|
| historical|      1052|
+-----------+----------+

+-----------+----------+
|source_load|total_rows|
+-----------+----------+
|incremental|      1041|
| historical|      1043|
+-----------+----------+

+-----------+----------+
|source_load|total_rows|
+-----------+----------+
|incremental|      1000|
| historical|      1002|
+-----------+----------+

